In [126]:
import pandas as pd
import psycopg2
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from io import StringIO
import joblib

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="ncaa"
)

cur = conn.cursor()

In [127]:
matchups_df = pd.read_sql("SELECT * FROM matchups_w_conferences", conn)

/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_82376/3137357462.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  matchups_df = pd.read_sql("SELECT * FROM matchups_w_conferences", conn)


In [129]:
def get_NET_ratings():
    url = "https://www.ncaa.com/rankings/basketball-men/d1/ncaa-mens-basketball-net-rankings"
    r = requests.get(url)
    soup = BeautifulSoup(r.content, 'html.parser')
    table = soup.find('table')
    df = pd.read_html(StringIO(str(table)))[0]
    df["Quad 1 W"] = df["Quad 1"].str.extract(r'(\d+)-\d+').astype(int)
    df["Quad 1 L"] = df["Quad 1"].str.extract(r'\d+-(\d+)').astype(int)
    df["Quad 2 W"] = df["Quad 2"].str.extract(r'(\d+)-\d+').astype(int) 
    df["Quad 2 L"] = df["Quad 2"].str.extract(r'\d+-(\d+)').astype(int)
    df["Quad 3 W"] = df["Quad 3"].str.extract(r'(\d+)-\d+').astype(int) 
    df["Quad 3 L"] = df["Quad 3"].str.extract(r'\d+-(\d+)').astype(int)
    df["Quad 4 W"] = df["Quad 4"].str.extract(r'(\d+)-\d+').astype(int) 
    df["Quad 4 L"] = df["Quad 4"].str.extract(r'\d+-(\d+)').astype(int)
    df["W"] = df["Record"].str.extract(r'(\d+)-\d+').astype(int)
    df["L"] = df["Record"].str.extract(r'\d+-(\d+)').astype(int)
    df["Road W"] = df["Road"].str.extract(r'(\d+)-\d+').astype(int)
    df["Road L"] = df["Road"].str.extract(r'\d+-(\d+)').astype(int)
    df["Home W"] = df["Home"].str.extract(r'(\d+)-\d+').astype(int)
    df["Home L"] = df["Home"].str.extract(r'\d+-(\d+)').astype(int)
    df["Neutral W"] = df["Neutral"].str.extract(r'(\d+)-\d+').astype(int)
    df["Neutral L"] = df["Neutral"].str.extract(r'\d+-(\d+)').astype(int)
    df["Non-Div I W"] = df["Non-Div I"].str.extract(r'(\d+)-\d+').astype(int)
    df["Non-Div I L"] = df["Non-Div I"].str.extract(r'\d+-(\d+)').astype(int)
    df.drop(columns=["Quad 1", "Quad 2", "Quad 3", "Quad 4", "Home", "Road", "Neutral", "Record", "Non-Div I"], inplace=True)
    return df

net_ratings = get_NET_ratings()

net_ratings = net_ratings.rename(columns={
        "Rank" : "net_rank",
        "School" : "school",
        "Conference" : "conference",
        "Previous" : "previous",
        "Quad 1 W" : "quad1_w",
        "Quad 1 L" : "quad1_l",
        "Quad 2 W" : "quad2_w",
        "Quad 2 L" : "quad2_l",
        "Quad 3 W" : "quad3_w",
        "Quad 3 L" : "quad3_l",
        "Quad 4 W" : "quad4_w",
        "Quad 4 L" : "quad4_l",
        "W" : "w", 
        "L" : "l",
        "Road W" : "road_w",
        "Road L" : "road_l",
        "Home W" : "home_w",
        "Home L" : "home_l",
        "Neutral W" : "neutral_w",
        "Neutral L" : "neutral_l",
        "Non-Div I W" : "non_div1_w",
        "Non-Div I L" : "non_div1_l"
    })

net_ratings['school'] = net_ratings['school'].str.replace(' St.', ' State')
net_ratings['school'] = net_ratings['school'].str.replace(' Ky.', ' Kentucky')
net_ratings['school'] = net_ratings['school'].str.replace('Ga.', 'Georgia')
net_ratings['school'] = net_ratings['school'].str.replace('Fla.', 'Florida')
net_ratings['school'] = net_ratings['school'].str.replace('Mich.', 'Michigan')
    
name_mapping = {
    "North Carolina" : "UNC",
    "Southern California" : "USC",
    "McNeese" : "McNeese State",
    "Seattle U" : "Seattle",
    "UNI" : "Northern Iowa",
    "Pittsburgh" : "Pitt",
    "Middle Tenn." : "Middle Tennessee",
    "Saint Mary's (CA)" : "Saint Mary's",
    "UC San Diego" : "UC-San Diego",
    "SFA" : "Stephen F. Austin",
    "LMU (CA)" : "Loyola Marymount",
    "UNCW" : "UNC Wilmington",
    "UC Irvine" : "UC-Irvine",
    "UC Davis" : "UC-Davis",
    "St. Thomas (MN)" : "St. Thomas",
    "Southern Ill." : "Southern Illinois",
    "Northern Colo." : "Northern Colorado",
    "UC Santa Barbara" : "UCSB",
    "Southern Miss." : "Southern Miss",
    "UT Martin" : "UT-Martin",
    "Col. of Charleston" : "College of Charleston",
    "Massachusetts" : "UMass",
    "FIU" : "Florida International",
    "UTRGV" : "Texas-Rio Grande Valley",
    "UIW" : "Incarnate Word",
    "Charleston So." : "Charleston Southern",
    "Nicholls" : "Nicholls State",
    "A&M-Corpus Christi" : "Texas A&M-Corpus Christi",
    "FGCU" : "Florida Gulf Coast",
    "Southeast Mo. State" : "Southeast Missouri State",
    "CSUN" : "Cal State Northridge",
    "Central Ark." : "Central Arkansas",
    "App State" : "Appalachian State",
    "Central Conn. State" : "Central Connecticut",
    "Lamar University" : "Lamar",
    "Saint Joseph's" : "St. Joseph's",
    "UC Riverside" : "UC-Riverside",
    "SIUE" : "SIU-Edwardsville",
    "Northern Ariz." : "Northern Arizona",
    "Boston U." : "Boston University",
    "Eastern Wash." : "Eastern Washington",
    "N.C. A&T" : "North Carolina A&T",
    "Saint Peter's" : "St. Peter's",
    "Western Caro." : "Western Carolina",
    "Southeastern La." : "Southeastern Louisiana",
    "Army West Point" : "Army",
    "Southern U." : "Southern",
    "Ark.-Pine Bluff" : "Arkansas-Pine Bluff",
    "North Ala." : "North Alabama",
    "CSU Bakersfield" : "Cal State Bakersfield",
    "Eastern Ill." : "Eastern Illinois",
    "Loyola Chicago" : "Loyola (IL)",
    "Alcorn" : "Alcorn State",
    "Mount State Mary's" : "Mount St. Mary's",
    "NIU" : "Northern Illinois",
    "UAlbany" : "Albany (NY)",
    "UMass Lowell" : "UMass-Lowell",
    "IU Indy" : "IU Indianapolis",
    "UMES" : "Maryland-Eastern Shore",
    "Southern Ind." : "Southern Indiana",
    "Western Ill." : "Western Illinois",
    "Loyola Maryland" : "Loyola (MD)",
    "N.C. Central" : "North Carolina Central",
    "ULM" : "Louisiana-Monroe",
    "Saint Francis" : "Saint Francis (PA)",
    "Mississippi Val." : "Mississippi Valley State"
}

net_ratings['school'] = net_ratings['school'].replace(name_mapping)

In [130]:
drop_cols = [
    "gameid", "team_games_before", 
    "opponent_games_before"
]

df = matchups_df.drop(columns=drop_cols)

In [131]:
df = df.merge(
    net_ratings[["school", "net_rank"]],
    left_on="team",
    right_on="school",
    how="left"
)

df = df[~df.school.isna()].drop(columns=["school"]).rename(columns={"net_rank" : "team_net_rank"})

df = df.merge(
    net_ratings[["school", "net_rank"]],
    left_on="opponent", 
    right_on="school",
    how="left"
)

df = df[~df.school.isna()].drop(columns=["school"]).rename(columns={"net_rank" : "opponent_net_rank"})

df

,team,opponent,date,location,opponent_location,team_poss,opponent_poss,team_ortg,opponent_ortg,team_drtg,...,team_fta,opponent_fta,team_fg2m,opponent_fg2m,team_fg2a,opponent_fg2a,team_conference,opp_conference,team_net_rank,opponent_net_rank
0,Belmont,Air Force,2025-11-03,home,away,67.875,61.275,116.390424,102.815177,102.815177,...,25,9,20,16,33,26,MVC,Mountain West,62.0,319.0
1,James Madison,Akron,2025-11-03,away,home,75.175,72.775,94.446292,116.798351,116.798351,...,13,29,13,19,32,38,Sun Belt,MAC,219.0,51.0
2,North Dakota,Alabama,2025-11-03,away,home,76.075,77.825,81.498521,116.929007,116.929007,...,17,27,19,23,42,31,Summit,SEC,300.0,12.0
3,Marquette,Albany (NY),2025-11-03,home,away,76.525,79.650,104.541000,66.541117,66.541117,...,39,14,19,14,34,37,Big East,America East,174.0,326.0
4,Wake Forest,American,2025-11-03,home,away,77.400,76.550,113.695090,96.668844,96.668844,...,24,18,24,17,40,28,ACC,Patriot League,61.0,240.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5087,IU Indianapolis,Youngstown State,2025-12-06,home,away,67.025,68.550,82.058933,113.785558,113.785558,...,19,18,15,23,33,35,Horizon,Horizon,325.0,182.0
5088,Robert Morris,Youngstown State,2025-12-17,home,away,63.975,66.450,125.048847,115.876599,115.876599,...,21,22,20,15,45,19,Horizon,Horizon,162.0,182.0
5089,USC Upstate,Youngstown State,2025-12-20,away,home,72.875,69.775,89.193825,106.055177,106.055177,...,25,29,17,15,38,31,Big South,Horizon,250.0,182.0
5090,Detroit Mercy,Youngstown State,2025-12-29,away,home,69.875,72.500,104.472272,93.793103,93.793103,...,25,20,17,12,41,29,Horizon,Horizon,266.0,182.0


In [132]:
features = pd.DataFrame()

features["team_conference"] = df['team_conference']

features["opp_conference"] = df['opp_conference']

features["date"]= df["date"]

features["team_pts"] = (
    df.team_pts
)

features["team_net_rank"] = (
    df.team_net_rank
)

features["opponent_net_rank"] = (
    df.opponent_net_rank
)

features["diff_net_rank"] = (
    df.team_net_rank - df.opponent_net_rank
)

features["team_poss_pre_game"] = (
    df
    .sort_values(["team", "date"])
    .groupby("team")
    .team_poss
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["team_poss_last_3"] = (
    df
    .sort_values(["team", "date"])
    .groupby("team")
    .team_poss
    .rolling(window=3)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["team_poss_last_5"] = (
    df
    .sort_values(["team", "date"])
    .groupby("team")
    .team_poss
    .rolling(window=5)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["opponent_poss_pre_game"] = (
    df
    .sort_values(["opponent", "date"])
    .groupby("opponent")
    .opponent_poss
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["opponent_poss_last_3"] = (
    df
    .sort_values(["opponent", "date"])
    .groupby("opponent")
    .opponent_poss
    .rolling(window=3)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["opponent_poss_last_5"] = (
    df
    .sort_values(["opponent", "date"])
    .groupby("opponent")
    .opponent_poss
    .rolling(window=5)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["team_ortg_pre_game"] = (
    df
    .sort_values(["team", "date"])
    .groupby("team")
    .team_ortg
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["team_ortg_last_3"] = (
    df
    .sort_values(["team", "date"])
    .groupby("team")
    .team_ortg
    .rolling(window=3)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["team_ortg_last_5"] = (
    df
    .sort_values(["team", "date"])
    .groupby("team")
    .team_ortg
    .rolling(window=5)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["opponent_drtg_pre_game"] = (
    df
    .sort_values(["opponent", "date"])
    .groupby("opponent")
    .opponent_drtg
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["opponent_drtg_last_3"] = (
    df
    .sort_values(["opponent", "date"])
    .groupby("opponent")
    .opponent_drtg
    .rolling(3)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["opponent_drtg_last_5"] = (
    df
    .sort_values(["opponent", "date"])
    .groupby("opponent")
    .opponent_drtg
    .rolling(5)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

features["location"] = (
    df.location.map({"home": 1, "away": 0})
)


In [133]:
df_sorted = df.sort_values("date").copy()

conf_daily_avg = (
    df_sorted
    .groupby(["team_conference", "date"], as_index=False)
    .team_pts
    .mean()
)

conf_daily_avg["team_conf_avg_pts_pre_game"] = (
    conf_daily_avg
    .sort_values("date")
    .groupby("team_conference", group_keys=False)
    .apply(lambda g: g["team_pts"].expanding().mean().shift(1))
)

features = features.merge(
    conf_daily_avg[["team_conference", "date", "team_conf_avg_pts_pre_game"]],
    on=["team_conference", "date"],
    how="left"
)

conf_daily_avg = (
    df_sorted
    .groupby(["opp_conference", "date"], as_index=False)
    .opponent_pts
    .mean()
)

conf_daily_avg["opp_conf_avg_pts_pre_game"] = (
    conf_daily_avg
    .sort_values("date")
    .groupby("opp_conference", group_keys=False)
    .apply(lambda g: g["opponent_pts"].expanding().mean().shift(1))
)

features = features.merge(
    conf_daily_avg[["opp_conference", "date", "opp_conf_avg_pts_pre_game"]],
    on=["opp_conference", "date"],
    how="left"
)

/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_82376/2052521066.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["team_pts"].expanding().mean().shift(1))
/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_82376/2052521066.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["opponent_pts"].expanding().mean().shift(1))


In [134]:
features = features.drop(["team_conference", "opp_conference", "date"], axis=1)

In [135]:
features.columns

Index(['team_pts', 'team_net_rank', 'opponent_net_rank', 'diff_net_rank',
       'team_poss_pre_game', 'team_poss_last_3', 'team_poss_last_5',
       'opponent_poss_pre_game', 'opponent_poss_last_3',
       'opponent_poss_last_5', 'team_ortg_pre_game', 'team_ortg_last_3',
       'team_ortg_last_5', 'opponent_drtg_pre_game', 'opponent_drtg_last_3',
       'opponent_drtg_last_5', 'location', 'team_conf_avg_pts_pre_game',
       'opp_conf_avg_pts_pre_game'],
      dtype='object')

In [136]:
def make_quantile_model(alpha):
    return HistGradientBoostingRegressor(
        loss="quantile",
        quantile=alpha,
        learning_rate=0.05,
        max_depth=6,
        max_iter=600,
        min_samples_leaf=25,
        l2_regularization=0.3,
        early_stopping=True,
        random_state=42
    )

cv = KFold(n_splits=5, shuffle=True, random_state=42)

X = features.drop(columns=["team_pts"])
y = features["team_pts"]

def cv_quantile_preds(alpha):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", make_quantile_model(alpha))
    ])
    return cross_val_predict(
        pipe,
        X,
        np.log1p(y),
        cv=cv,
    )

q5_log = cv_quantile_preds(0.05)
q10_log = cv_quantile_preds(0.10)
q25_log = cv_quantile_preds(0.25)
q50_log = cv_quantile_preds(0.50)
q75_log = cv_quantile_preds(0.75)
q90_log = cv_quantile_preds(0.90)
q95_log = cv_quantile_preds(0.95)

q5 = np.expm1(q5_log)
q10 = np.expm1(q10_log)
q25 = np.expm1(q25_log)
q50 = np.expm1(q50_log)
q75 = np.expm1(q75_log)
q90 = np.expm1(q90_log)
q95 = np.expm1(q95_log)

residual_50 = np.maximum(q25 - y, y - q75)
delta_50 = np.quantile(residual_50, 0.5)

residual_80 = np.maximum(q10 - y, y - q90)
delta_80 = np.quantile(residual_80, 0.8)

residual_90 = np.maximum(q5 - y, y - q95)
delta_90 = np.quantile(residual_90, 0.9)


joblib.dump(delta_50, "../backend/models/conformal_delta_50.joblib")
joblib.dump(delta_80, "../backend/models/conformal_delta_80.joblib")
joblib.dump(delta_90, "../backend/models/conformal_delta_90.joblib")

q25_adj = q25 - delta_50
q75_adj = q75 + delta_50

q10_adj = q10 - delta_80
q90_adj = q90 + delta_80

q5_adj = q5 - delta_90
q95_adj = q95 + delta_90

coverage_50 = np.mean((y >= q25_adj) & (y <= q75_adj))
print(f"Empirical coverage (25–75): {coverage_50:.3f}")

coverage_80 = np.mean((y >= q10_adj) & (y <= q90_adj))
print(f"Empirical coverage (10–90): {coverage_80:.3f}")

coverage_90 = np.mean((y >= q5_adj) & (y <= q95_adj))
print(f"Empirical coverage (5–95): {coverage_90:.3f}")

Empirical coverage (25–75): 0.500
Empirical coverage (10–90): 0.800
Empirical coverage (5–95): 0.900


In [137]:
def train_quantile_model(X, y, alpha):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingRegressor(
            loss="quantile",
            quantile=alpha,
            learning_rate=0.05,
            max_depth=6,
            max_iter=600,
            min_samples_leaf=25,
            l2_regularization=0.3,
            early_stopping=True,
            random_state=42
        ))
    ])
    
    pipe.fit(X, np.log1p(y))
    return pipe

X = features.drop(columns=["team_pts"])
y = features["team_pts"]

model_q5 = train_quantile_model(X, y, 0.05)
model_q10 = train_quantile_model(X, y, 0.10)
model_q25 = train_quantile_model(X, y, 0.25)
model_q50 = train_quantile_model(X, y, 0.50)
model_q75 = train_quantile_model(X, y, 0.75)
model_q90 = train_quantile_model(X, y, 0.90)
model_q95 = train_quantile_model(X, y, 0.95)

joblib.dump(model_q5, "../backend/models/model_q5.joblib")
joblib.dump(model_q10, "../backend/models/model_q10.joblib")
joblib.dump(model_q25, "../backend/models/model_q25.joblib")
joblib.dump(model_q50, "../backend/models/model_q50.joblib")
joblib.dump(model_q75, "../backend/models/model_q75.joblib")
joblib.dump(model_q90, "../backend/models/model_q90.joblib")
joblib.dump(model_q95, "../backend/models/model_q95.joblib")

['../backend/models/model_q95.joblib']